### **Custom tool**

In [1]:
from langchain_core.tools import tool

This @tool decorator is the simplest way to define a custom tool. The decorator uses the function name as the tool name by default, but this can be overridden by passing a string as the first argument. Additionally, the decorator will use the function's docstring as the tool's description - so a docstring MUST be provided.

In [2]:
@tool
def division(a:int, b:int)->int:
    """Divide 2 numbers"""
    return a/b

In [3]:
print(division.name)
print(division.description)
print(division.args)

division
Divide 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


An `async` function allows your program to do other work while waiting for slow operations like API calls, database queries, file I/O, or network requests to complete.

In [4]:
@tool
async def amultiply(a:int, b:int)->int:
    """Multiply two numbers"""
    return a*b

In [5]:
from typing import Annotated, List

In [6]:
@tool
def multiply_max(
    a:Annotated[int, "A value"],
    b:Annotated[List[int], "List of values"]
    ) -> int:
    """Multipy by maximum of b"""
    return a*max(b)

In [7]:
print(multiply_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'List of values', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


### ***Structured Tool***

The StructuredTool.from_function class method provides a bit more configurability than the `@tool` decorator, without requiring much additional code.

In [8]:
from langchain_core.tools import StructuredTool

In [9]:
def multiply(a:int, b:int)->int:
    """Multiply two numbers"""
    return a*b

async def a_multiply(a:int, b:int)->int:
    """Multipy two numbers"""
    return a*b

calculator = StructuredTool.from_function(func=multiply, coroutine=a_multiply)

print(calculator.invoke({"a":2, "b": 5}))
print(await calculator.ainvoke({"a":5, "b":12}))

10
60


In [10]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\hrida\AppData\Local\Temp\ipykernel_17320\2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [24]:
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=500)
search_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
print(search_tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vec


In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ["OPENROUTER_API_KEY"]

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="liquid/lfm-2.5-1.2b-thinking:free",  
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [29]:
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

agent = llm.bind_tools([calculator, search_tool])

tools = {
    search_tool.name: search_tool,
    calculator.name: calculator
}

messages = [
    HumanMessage(
        content="What is the population of Delhi multiplied by 3?"
    )
]

response = agent.invoke(messages)

print(response.tool_calls)
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
print("Calling tool:", tool_name)
print("Args:", tool_call["args"])

result = tools[tool_call["name"]].invoke(tool_call["args"])

messages.append(response)

messages.append(
    ToolMessage(
        content=result,
        tool_call_id=tool_call["id"]
    )
)

response = agent.invoke(messages)

print(response.tool_calls)
tool_call = response.tool_calls[0]
tool_name = tool_call["name"]
print("Calling tool:", tool_name)
print("Args:", tool_call["args"])

result = tools[tool_call["name"]].invoke(
    tool_call["args"]
)

messages.append(response)

messages.append(ToolMessage(
    content=str(result),
    tool_call_id = tool_call["id"]
))

response = agent.invoke(messages)

print(response.content)



[{'name': 'multiply', 'args': {'a': 32280000, 'b': 3}, 'id': 'call_40b1e722aaa84ae7be0fec50', 'type': 'tool_call'}]
Calling tool: multiply
Args: {'a': 32280000, 'b': 3}
[{'name': 'multiply', 'args': {'a': 96840000, 'b': 3}, 'id': 'call_d997ac006de542e1b923cf0a', 'type': 'tool_call'}]
Calling tool: multiply
Args: {'a': 96840000, 'b': 3}
290520000
